In [ ]:
import csv
import hashlib
import importlib
import json
import platform
import shutil
import subprocess
import sys
import time
from importlib.metadata import version
from pathlib import Path

NOTEBOOK = "07_full_history_coverage_and_scale_plan.ipynb"
notebook_started = time.perf_counter()

candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
REPO_ROOT = next(
    (
        path for path in candidates
        if (path / "pyproject.toml").is_file()
        and (path / "configs/research_config.example.json").is_file()
        and (path / "src/eurusd_research/quality.py").is_file()
    ),
    None,
)
if REPO_ROOT is None:
    entered = input("Full repository folder containing configs and src: ").strip().strip('"')
    if not entered:
        raise ValueError("Enter the full repository folder path.")
    REPO_ROOT = Path(entered).expanduser().resolve()

if not (REPO_ROOT / "src/eurusd_research/quality.py").is_file():
    raise RuntimeError("The selected repository is incomplete.")

import numpy as np
import pandas as pd

from eurusd_research.config import load_config
from eurusd_research import features as feature_io
from eurusd_research.data import _sha256, _stamp
from eurusd_research.quality import EXPECTED_COLUMNS

config = load_config(REPO_ROOT)

def matching_sources(folder):
    try:
        return sorted(
            path.resolve()
            for path in Path(folder).glob("EURUSD_Ticks_*.csv")
            if path.is_file()
        )
    except OSError:
        return []

raw_candidates = []
for folder in (REPO_ROOT / "data", Path(config["raw_data_dir"])):
    folder = folder.expanduser().resolve()
    if folder not in raw_candidates and matching_sources(folder):
        raw_candidates.append(folder)

if len(raw_candidates) == 1:
    RAW_ROOT = raw_candidates[0]
elif len(raw_candidates) > 1:
    print("Multiple raw-data folders contain matching exports:")
    for folder in raw_candidates:
        print(folder)
    entered = input("Full raw-data folder to audit: ").strip().strip('"')
    if not entered:
        raise ValueError("Select the intended raw-data folder.")
    RAW_ROOT = Path(entered).expanduser().resolve()
else:
    entered = input("Full folder containing the EURUSD CSV exports: ").strip().strip('"')
    if not entered:
        raise ValueError("Enter the raw-data folder path.")
    RAW_ROOT = Path(entered).expanduser().resolve()

source_files = matching_sources(RAW_ROOT)
if not source_files:
    raise RuntimeError("No EURUSD_Ticks_*.csv files were found.")

audit_settings = config.get("full_history_audit", {
    "checkpoint_batches": 16,
    "progress_batches": 128,
    "max_gap_examples": 20,
    "max_unassigned_examples": 5,
    "maximum_dataframe_mib": 128,
    "maximum_source_files": 20,
    "maximum_total_source_gib": 100,
    "output_estimate_multiplier": 2.0,
    "scratch_reserve_gib": 8,
    "planning_training_windows_days": [30, 90, 180, 365],
})

audit_parameters = {
    "timezone": config["source"]["source_timezone"],
    "timestamp_format": config["source"]["timestamp_format"],
    "pip_size": config["pip_size"],
    "gap_seconds": config["sampling_features"]["substantial_gap_seconds"],
    "extreme_spread_pips": config["ingestion"]["extreme_spread_audit_pips"],
    "block_bytes": config["ingestion"]["csv_block_bytes"],
    "checkpoint_batches": audit_settings["checkpoint_batches"],
    "progress_batches": audit_settings["progress_batches"],
    "max_gap_examples": audit_settings["max_gap_examples"],
    "max_unassigned_examples": audit_settings["max_unassigned_examples"],
    "maximum_dataframe_mib": audit_settings["maximum_dataframe_mib"],
}

versions = {
    name: version(name)
    for name in ("numpy", "pandas", "pyarrow", "pytest", "tzdata", "pytz")
}
new_sources = {}

print("NOTEBOOK_07_PREFLIGHT")
print("Repository:", REPO_ROOT)
print("Raw-data folder:", RAW_ROOT)
print("Matching exports:", len(source_files))
print("Source timezone interpretation:", audit_parameters["timezone"])
print("No package installation or restart is required.")

In [ ]:
inventory_records = []
initial_stamps = {}

for path in source_files:
    initial_stamps[path.name] = _stamp(path)

    with path.open("r", encoding="utf-8-sig", newline="") as handle:
        reader = csv.reader(handle)
        header = next(reader, None)
        first_rows = []
        for _ in range(2):
            row = next(reader, None)
            if row is None:
                break
            first_rows.append(row)

    if header != EXPECTED_COLUMNS:
        raise RuntimeError(f"Unexpected header in {path.name}: {header}")
    if any(len(row) != len(EXPECTED_COLUMNS) for row in first_rows):
        raise RuntimeError(f"Malformed prefix record in {path.name}.")
    if not first_rows:
        raise RuntimeError(f"No data records found in {path.name}.")

    inventory_records.append({
        "source_file": path.name,
        "bytes": path.stat().st_size,
        "header_verified": True,
        "first_source_timestamp_text": first_rows[0][0],
    })

inventory_table = pd.DataFrame(inventory_records)
total_source_bytes = int(inventory_table["bytes"].sum())

if len(source_files) > audit_settings["maximum_source_files"]:
    raise RuntimeError("Source count exceeds the explicit audit limit.")
if total_source_bytes > audit_settings["maximum_total_source_gib"] * 1024**3:
    raise RuntimeError("Source bytes exceed the explicit audit limit.")

print("SOURCE_INVENTORY")
print(inventory_table.to_string(index=False))
print("Total source GiB:", round(total_source_bytes / 1024**3, 3))
print("PASS: selected source headers and prefix field counts agree.")

In [ ]:
new_sources["src/eurusd_research/coverage_audit.py"] = r'''
"""Bounded, resumable full-source coverage and quote-quality auditing."""

import hashlib
import json
import platform
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.csv as arrow_csv

from . import features as io
from .data import _sha256, _stamp, _writer_lock
from .quality import EXPECTED_COLUMNS, normalise_quotes


COUNT_COLUMNS = [
    "quote_rows", "price_eligible_quote_rows", "size_eligible_quote_rows",
    "invalid_price_rows", "crossed_quote_rows", "locked_quote_rows",
    "missing_size_rows", "invalid_size_rows", "negative_size_rows",
    "zero_size_rows", "wide_spread_rows",
    "adjacent_exact_export_duplicate_rows",
    "source_order_backsteps", "gaps_above_threshold",
]
GAP_LABELS = [
    "0", "(0,1]", "(1,5]", "(5,30]", "(30,60]",
    "(60,300]", "(300,3600]", "(3600,86400]", ">86400",
]


def new_audit(source_name, source_sha256):
    return {
        "source_name": source_name, "source_sha256": source_sha256,
        "rows_scanned": 0, "valid_timestamp_rows": 0,
        "unassigned_timestamp_rows": 0,
        "price_eligible_quote_rows": 0, "size_eligible_quote_rows": 0,
        "next_batch": 0, "previous_utc": None, "previous_raw": None,
        "previous_valid_utc": None, "previous_valid_row": None,
        "utc_min": None, "utc_max": None,
        "first_record": None, "last_record": None,
        "flags": {}, "daily": {},
        "adjacent_exact_export_duplicate_rows": 0,
        "source_order_backsteps": 0, "repeated_valid_timestamp_steps": 0,
        "gaps_above_threshold": 0, "gaps_crossing_unassigned_rows": 0,
        "maximum_source_sequence_gap_seconds": 0.0,
        "gap_histogram": [0] * len(GAP_LABELS),
        "largest_gaps": [], "unassigned_examples": [],
        "maximum_observed_dataframe_bytes": 0,
    }


def _time_bound(mapping, key, candidate, minimum):
    if pd.isna(candidate):
        return
    candidate = pd.Timestamp(candidate)
    existing = mapping.get(key)
    if existing is None or (
        candidate < pd.Timestamp(existing) if minimum
        else candidate > pd.Timestamp(existing)
    ):
        mapping[key] = candidate.isoformat()


def _number_bound(mapping, key, candidate, minimum):
    if pd.isna(candidate):
        return
    candidate = float(candidate)
    existing = mapping.get(key)
    if existing is None or (
        candidate < existing if minimum else candidate > existing
    ):
        mapping[key] = candidate


def consume_rows(state, raw, parameters):
    raw = raw.reset_index(drop=True).fillna("")
    if raw.empty:
        return
    frame = normalise_quotes(
        raw,
        source_name=state["source_name"],
        source_sha256=state["source_sha256"],
        first_row=state["rows_scanned"] + 1,
        previous_utc=state["previous_utc"],
        **{
            key: parameters[key] for key in (
                "timezone", "timestamp_format", "pip_size",
                "gap_seconds", "extreme_spread_pips",
            )
        },
    )
    observed_bytes = int(
        raw.memory_usage(index=True, deep=True).sum()
        + frame.memory_usage(index=True, deep=True).sum()
    )
    if observed_bytes > parameters["maximum_dataframe_mib"] * 1024**2:
        raise RuntimeError("Audit dataframes exceed their configured memory guard.")
    state["maximum_observed_dataframe_bytes"] = max(
        state["maximum_observed_dataframe_bytes"], observed_bytes
    )

    flag_names = [name for name in frame if name.startswith("flag_")]
    for name in flag_names:
        state["flags"][name] = state["flags"].get(name, 0) + int(frame[name].sum())

    duplicate = raw.eq(raw.shift()).all(axis=1)
    if state["previous_raw"] is not None:
        duplicate.iloc[0] = raw.iloc[0].tolist() == state["previous_raw"]

    times = frame["timestamp_utc"]
    assigned = times.notna()
    price = assigned & ~frame["flag_invalid_price"] & ~frame["flag_crossed"]
    total_size = frame["ask_size"] + frame["bid_size"]
    size = (
        price & np.isfinite(frame["ask_size"]) & np.isfinite(frame["bid_size"])
        & frame["ask_size"].ge(0) & frame["bid_size"].ge(0)
        & np.isfinite(total_size) & total_size.gt(0)
    )

    valid_times = times.loc[assigned]
    valid_rows = frame.loc[assigned, "source_row"]
    previous_times = valid_times.shift()
    previous_rows = valid_rows.shift()
    if len(valid_times) and state["previous_valid_utc"] is not None:
        previous_times.iloc[0] = pd.Timestamp(state["previous_valid_utc"])
        previous_rows.iloc[0] = state["previous_valid_row"]
    differences = (valid_times - previous_times).dt.total_seconds()
    gap = differences.gt(parameters["gap_seconds"])
    backstep = differences.lt(0)

    nonnegative = differences.loc[differences.ge(0)].to_numpy(dtype=float)
    bins = np.searchsorted(
        [0, 1, 5, 30, 60, 300, 3600, 86400], nonnegative, side="left"
    )
    histogram = np.bincount(bins, minlength=len(GAP_LABELS))
    state["gap_histogram"] = [
        int(a + b) for a, b in zip(state["gap_histogram"], histogram)
    ]
    state["source_order_backsteps"] += int(backstep.sum())
    state["repeated_valid_timestamp_steps"] += int(differences.eq(0).sum())
    state["gaps_above_threshold"] += int(gap.sum())
    crossing_unassigned = valid_rows.sub(previous_rows).gt(1)
    state["gaps_crossing_unassigned_rows"] += int((gap & crossing_unassigned).sum())
    if len(nonnegative):
        state["maximum_source_sequence_gap_seconds"] = max(
            state["maximum_source_sequence_gap_seconds"], float(nonnegative.max())
        )

    # Only bounded diagnostic examples are iterated.
    largest = differences.loc[gap].nlargest(parameters["max_gap_examples"])
    for index, seconds in largest.items():
        state["largest_gaps"].append({
            "start_utc": previous_times.loc[index].isoformat(),
            "end_utc": valid_times.loc[index].isoformat(),
            "start_source_row": int(previous_rows.loc[index]),
            "end_source_row": int(valid_rows.loc[index]),
            "seconds": float(seconds),
            "crosses_unassigned_rows": bool(crossing_unassigned.loc[index]),
        })
    state["largest_gaps"] = sorted(
        state["largest_gaps"],
        key=lambda item: (-item["seconds"], item["end_source_row"]),
    )[:parameters["max_gap_examples"]]

    remaining = parameters["max_unassigned_examples"] - len(state["unassigned_examples"])
    for index in frame.index[~assigned][:max(0, remaining)]:
        state["unassigned_examples"].append({
            "source_row": int(frame.loc[index, "source_row"]),
            "raw_fields": raw.loc[index].tolist(),
            "reasons": [
                name for name in (
                    "flag_missing_timestamp", "flag_unparseable_timestamp",
                    "flag_unresolved_local_time",
                ) if bool(frame.loc[index, name])
            ],
        })

    data = pd.DataFrame({
        "date_utc": times.dt.floor("D"),
        "timestamp_utc": times,
        "quote_rows": np.ones(len(frame), dtype=np.int64),
        "price_eligible_quote_rows": price,
        "size_eligible_quote_rows": size,
        "invalid_price_rows": frame["flag_invalid_price"],
        "crossed_quote_rows": frame["flag_crossed"],
        "locked_quote_rows": frame["flag_locked"],
        "missing_size_rows": frame["flag_missing_size"],
        "invalid_size_rows": frame["flag_invalid_size"],
        "negative_size_rows": frame["flag_negative_size"],
        "zero_size_rows": frame["flag_zero_size"],
        "wide_spread_rows": frame["flag_extreme_spread"],
        "adjacent_exact_export_duplicate_rows": duplicate,
        "source_order_backsteps": backstep.reindex(frame.index, fill_value=False),
        "gaps_above_threshold": gap.reindex(frame.index, fill_value=False),
        "spread_pips": frame["spread_pips"],
        "gap_seconds": differences.where(differences.ge(0)).reindex(frame.index),
    }).dropna(subset=["date_utc"])

    aggregates = {
        name: (name, "sum") for name in COUNT_COLUMNS
    }
    aggregates.update({
        "first_quote_utc": ("timestamp_utc", "min"),
        "last_quote_utc": ("timestamp_utc", "max"),
        "min_spread_pips": ("spread_pips", "min"),
        "max_spread_pips": ("spread_pips", "max"),
        "max_gap_seconds": ("gap_seconds", "max"),
    })
    grouped = data.groupby("date_utc", sort=False).agg(**aggregates).reset_index()
    for record in grouped.to_dict(orient="records"):
        date = record["date_utc"].date().isoformat()
        day = state["daily"].setdefault(date, {
            "date_utc": date,
            **{name: 0 for name in COUNT_COLUMNS},
            "first_quote_utc": None, "last_quote_utc": None,
            "min_spread_pips": None, "max_spread_pips": None,
            "max_gap_seconds": None,
        })
        for name in COUNT_COLUMNS:
            day[name] += int(record[name])
        _time_bound(day, "first_quote_utc", record["first_quote_utc"], True)
        _time_bound(day, "last_quote_utc", record["last_quote_utc"], False)
        _number_bound(day, "min_spread_pips", record["min_spread_pips"], True)
        _number_bound(day, "max_spread_pips", record["max_spread_pips"], False)
        _number_bound(day, "max_gap_seconds", record["max_gap_seconds"], False)

    if len(valid_times):
        _time_bound(state, "utc_min", valid_times.min(), True)
        _time_bound(state, "utc_max", valid_times.max(), False)
        state["previous_valid_utc"] = valid_times.iloc[-1].isoformat()
        state["previous_valid_row"] = int(valid_rows.iloc[-1])

    def edge(index):
        timestamp = times.iloc[index]
        return {
            "source_row": int(frame["source_row"].iloc[index]),
            "timestamp_utc": None if pd.isna(timestamp) else timestamp.isoformat(),
            "raw_fields": raw.iloc[index].tolist(),
        }

    if state["first_record"] is None:
        state["first_record"] = edge(0)
    state["last_record"] = edge(-1)
    state["previous_raw"] = raw.iloc[-1].tolist()
    state["previous_utc"] = None if pd.isna(times.iloc[-1]) else times.iloc[-1].isoformat()
    state["rows_scanned"] += len(frame)
    state["valid_timestamp_rows"] += int(assigned.sum())
    state["unassigned_timestamp_rows"] += int((~assigned).sum())
    state["price_eligible_quote_rows"] += int(price.sum())
    state["size_eligible_quote_rows"] += int(size.sum())
    state["adjacent_exact_export_duplicate_rows"] += int(duplicate.sum())
    state["next_batch"] += 1
'''.lstrip()

print("Prepared streaming coverage and quality aggregation.")

new_sources["src/eurusd_research/coverage_audit.py"] += r'''


def daily_frame(state):
    columns = [
        "date_utc", *COUNT_COLUMNS,
        "first_quote_utc", "last_quote_utc",
        "min_spread_pips", "max_spread_pips", "max_gap_seconds",
    ]
    frame = pd.DataFrame(list(state["daily"].values()), columns=columns)
    for name in ("date_utc", "first_quote_utc", "last_quote_utc"):
        frame[name] = pd.to_datetime(
            frame[name], format="ISO8601", utc=True
        ).astype("datetime64[us, UTC]")
    for name in COUNT_COLUMNS:
        frame[name] = frame[name].astype("int64")
    for name in ("min_spread_pips", "max_spread_pips", "max_gap_seconds"):
        frame[name] = frame[name].astype("float64")
    frame["source_file"] = pd.Series(
        state["source_name"], index=frame.index, dtype="string"
    )
    return frame.sort_values("date_utc").reset_index(drop=True)


def _verify_completed(state, root):
    path = (root / state["daily_output"]["path"]).resolve()
    if (
        not path.is_relative_to(root) or not path.is_file()
        or _sha256(path) != state["daily_output"]["sha256"]
    ):
        raise RuntimeError("Daily audit output checksum failed.")
    daily = pd.read_parquet(path)
    if (
        len(daily) != state["daily_output"]["rows"]
        or int(daily["quote_rows"].sum()) != state["valid_timestamp_rows"]
        or state["rows_scanned"] != (
            state["valid_timestamp_rows"] + state["unassigned_timestamp_rows"]
        )
        or int(daily["price_eligible_quote_rows"].sum())
            != state["price_eligible_quote_rows"]
    ):
        raise RuntimeError("Source audit row conservation failed.")
    unassigned_flags = sum(state["flags"].get(name, 0) for name in (
        "flag_missing_timestamp", "flag_unparseable_timestamp",
        "flag_unresolved_local_time",
    ))
    if unassigned_flags != state["unassigned_timestamp_rows"]:
        raise RuntimeError("Unassigned timestamp flags do not reconcile.")


def _register_source(manifest_root, state):
    key = hashlib.sha256(state["source_name"].encode()).hexdigest()[:24]
    path = manifest_root / f"source-{key}.json"
    registry = (
        json.loads(path.read_text(encoding="utf-8")) if path.exists()
        else {"source_name": state["source_name"], "completed_audits": []}
    )
    entry = {
        "job_id": state["job_id"],
        "source_sha256": state["source_sha256"],
    }
    if entry not in registry["completed_audits"]:
        registry["completed_audits"].append(entry)
        io.write_json(path, registry)
    return len({
        item["source_sha256"] for item in registry["completed_audits"]
    })


def audit_source(source_path, repository_root, parameters):
    source = Path(source_path).resolve()
    root = Path(repository_root).resolve()
    if not source.is_file():
        raise FileNotFoundError(source)
    if (
        parameters["block_bytes"] < 128
        or parameters["checkpoint_batches"] < 1
        or parameters["progress_batches"] < 1
        or parameters["max_gap_examples"] < 1
        or parameters["max_unassigned_examples"] < 0
    ):
        raise ValueError("Invalid audit resource settings.")

    stamp = _stamp(source)
    source_hash = _sha256(source)
    implementation = hashlib.sha256(b"".join(
        Path(__file__).with_name(name).read_bytes()
        for name in ("coverage_audit.py", "quality.py", "data.py", "features.py")
    )).hexdigest()
    identity = {
        "source_name": source.name,
        "source_sha256": source_hash,
        "source_bytes": stamp[0],
        "parameters": parameters,
        "implementation_sha256": implementation,
        "python": platform.python_version(),
        "versions": {
            name: version(name)
            for name in ("numpy", "pandas", "pyarrow", "tzdata", "pytz")
        },
    }
    job_id = hashlib.sha256(
        json.dumps(identity, sort_keys=True).encode()
    ).hexdigest()[:24]
    manifest_root = root / "artifacts/manifests/coverage_audit"
    manifest = manifest_root / f"job-{job_id}.json"

    with _writer_lock(manifest_root):
        if manifest.exists():
            state = json.loads(manifest.read_text(encoding="utf-8"))
            if state["identity"] != identity:
                raise RuntimeError("Existing source audit identity differs.")
            if state["status"] == "complete":
                _verify_completed(state, root)
                if _stamp(source) != stamp:
                    raise RuntimeError("Source changed while validating its cached audit.")
                versions_seen = _register_source(manifest_root, state)
                return {
                    **state, "cache_hit": True,
                    "recorded_source_content_versions": versions_seen,
                }
        else:
            state = {
                **new_audit(source.name, source_hash),
                "job_id": job_id, "status": "running", "identity": identity,
                "manifest_path": manifest.relative_to(root).as_posix(),
            }
            io.write_json(manifest, state)

        committed_batches = state["next_batch"]
        row_offset = 0
        with source.open("rb") as handle:
            if handle.read(3) != b"\xef\xbb\xbf":
                handle.seek(0)
            reader = arrow_csv.open_csv(
                handle,
                read_options=arrow_csv.ReadOptions(
                    block_size=parameters["block_bytes"], use_threads=False
                ),
                parse_options=arrow_csv.ParseOptions(
                    delimiter=",", newlines_in_values=True,
                    ignore_empty_lines=False,
                ),
                convert_options=arrow_csv.ConvertOptions(
                    column_types={name: pa.string() for name in EXPECTED_COLUMNS},
                    strings_can_be_null=False, check_utf8=True,
                ),
            )
            try:
                if reader.schema.names != EXPECTED_COLUMNS:
                    raise ValueError(f"Unexpected source schema: {reader.schema.names}")
                for number, batch in enumerate(reader):
                    if number < committed_batches:
                        row_offset += batch.num_rows
                        if number + 1 == committed_batches and row_offset != state["rows_scanned"]:
                            raise RuntimeError("Checkpoint replay row count differs.")
                        continue
                    if number != state["next_batch"] or row_offset != state["rows_scanned"]:
                        raise RuntimeError("Audit checkpoint position differs.")

                    consume_rows(state, batch.to_pandas(), parameters)
                    row_offset += batch.num_rows

                    if state["next_batch"] % parameters["checkpoint_batches"] == 0:
                        io.write_json(manifest, state)
                    if state["next_batch"] % parameters["progress_batches"] == 0:
                        print(f"{source.name}: {state['rows_scanned']:,} rows audited")
            finally:
                reader.close()

        if row_offset != state["rows_scanned"] or state["rows_scanned"] == 0:
            raise RuntimeError("Final audit row count is invalid.")
        if _stamp(source) != stamp or _sha256(source) != source_hash:
            raise RuntimeError("Source changed during auditing; review before resuming.")

        daily = daily_frame(state)
        output = root / "data/interim/coverage_audit" / job_id / "daily.parquet"
        record = io._save_frame(daily, output)
        state["daily_output"] = {
            **record, "path": output.relative_to(root).as_posix()
        }
        state["status"] = "complete"
        _verify_completed(state, root)
        io.write_json(manifest, state)
        versions_seen = _register_source(manifest_root, state)
        return {
            **state, "cache_hit": False,
            "recorded_source_content_versions": versions_seen,
        }


def source_overlap_ranges(states):
    records = []
    usable = [
        state for state in states
        if state["utc_min"] is not None and state["utc_max"] is not None
    ]
    for i, left in enumerate(usable):
        for right in usable[i + 1:]:
            start = max(pd.Timestamp(left["utc_min"]), pd.Timestamp(right["utc_min"]))
            end = min(pd.Timestamp(left["utc_max"]), pd.Timestamp(right["utc_max"]))
            if start <= end:
                records.append({
                    "left_source": left["source_name"],
                    "right_source": right["source_name"],
                    "overlap_start_utc": start.isoformat(),
                    "overlap_end_utc": end.isoformat(),
                    "overlap_seconds": float((end - start).total_seconds()),
                    "shared_boundary_timestamp_only": start == end,
                    "identical_records_proven": False,
                })
    return records
'''

print("Prepared resumable source scanning and overlap checks.")

new_sources["tests/test_coverage_audit.py"] = r'''
import copy

import pandas as pd
import pytest

from eurusd_research.coverage_audit import (
    audit_source, consume_rows, new_audit, source_overlap_ranges,
)


def parameters():
    return {
        "timezone": "UTC",
        "timestamp_format": "%Y.%m.%d %H:%M:%S.%f",
        "pip_size": 0.0001, "gap_seconds": 30,
        "extreme_spread_pips": 20, "block_bytes": 256,
        "checkpoint_batches": 2, "progress_batches": 100,
        "max_gap_examples": 20, "max_unassigned_examples": 5,
        "maximum_dataframe_mib": 20,
    }


def quotes(times):
    return pd.DataFrame({
        "Time (EET)": times,
        "Ask": ["1.1002"] * len(times),
        "Bid": ["1.1000"] * len(times),
        "AskVolume": ["1"] * len(times),
        "BidVolume": ["3"] * len(times),
    })


def test_batch_partition_consistency():
    raw = quotes([
        "2020.01.31 23:59:59.000",
        "2020.02.01 00:00:00.000",
        "2020.02.01 00:00:00.000",
        "bad",
        "2020.02.01 00:00:45.000",
        "2020.02.01 00:00:44.000",
        "2020.02.01 00:00:46.000",
    ])
    whole = new_audit("synthetic.csv", "synthetic")
    consume_rows(whole, raw, parameters())

    blocked = new_audit("synthetic.csv", "synthetic")
    for left, right in ((0, 2), (2, 4), (4, 7)):
        consume_rows(blocked, raw.iloc[left:right], parameters())

    for state in (whole, blocked):
        state.pop("next_batch")
        state.pop("maximum_observed_dataframe_bytes")
    assert whole == blocked
    assert whole["adjacent_exact_export_duplicate_rows"] == 1
    assert whole["source_order_backsteps"] == 1
    assert whole["gaps_crossing_unassigned_rows"] == 1
    assert whole["maximum_source_sequence_gap_seconds"] == 45.0


def test_missing_zero_and_negative_sizes_are_distinct():
    raw = quotes([
        "2020.01.01 00:00:00.000",
        "2020.01.01 00:00:01.000",
        "2020.01.01 00:00:02.000",
    ])
    raw["AskVolume"] = ["", "0", "-1"]
    state = new_audit("synthetic.csv", "synthetic")
    consume_rows(state, raw, parameters())
    assert state["flags"]["flag_missing_size"] == 1
    assert state["flags"]["flag_zero_size"] == 1
    assert state["flags"]["flag_negative_size"] == 1
    assert state["size_eligible_quote_rows"] == 1


def test_unresolved_dst_times_are_not_assigned():
    raw = quotes([
        "2016.03.27 03:30:00.000",
        "2016.10.30 03:30:00.000",
    ])
    settings = parameters()
    settings["timezone"] = "Europe/Helsinki"
    state = new_audit("synthetic.csv", "synthetic")
    consume_rows(state, raw, settings)
    assert state["unassigned_timestamp_rows"] == 2
    assert state["flags"]["flag_unresolved_local_time"] == 2


def test_all_pairs_detect_nested_source_overlaps():
    states = [
        {"source_name": "a", "utc_min": "2020-01-01T00:00:00Z", "utc_max": "2020-01-10T00:00:00Z"},
        {"source_name": "b", "utc_min": "2020-01-02T00:00:00Z", "utc_max": "2020-01-03T00:00:00Z"},
        {"source_name": "c", "utc_min": "2020-01-05T00:00:00Z", "utc_max": "2020-01-06T00:00:00Z"},
    ]
    overlaps = source_overlap_ranges(states)
    assert len(overlaps) == 2
    assert all(not item["identical_records_proven"] for item in overlaps)


def test_completed_cache_and_changed_source_version(tmp_path):
    raw = quotes([
        "2020.01.01 00:00:00.000",
        "2020.01.01 00:00:01.000",
        "2020.01.01 00:00:02.000",
    ])
    source = tmp_path / "source.csv"
    root = tmp_path / "project"
    raw.to_csv(source, index=False)

    first = audit_source(source, root, parameters())
    second = audit_source(source, root, parameters())
    assert first["job_id"] == second["job_id"]
    assert second["cache_hit"]
    assert second["rows_scanned"] == 3

    raw.loc[0, "Ask"] = "1.1003"
    raw.to_csv(source, index=False)
    changed = audit_source(source, root, parameters())
    assert changed["job_id"] != first["job_id"]
    assert changed["recorded_source_content_versions"] == 2


def test_changed_audit_output_is_detected(tmp_path):
    source = tmp_path / "source.csv"
    quotes(["2020.01.01 00:00:00.000"]).to_csv(source, index=False)
    root = tmp_path / "project"
    state = audit_source(source, root, parameters())
    (root / state["daily_output"]["path"]).write_bytes(b"changed")
    with pytest.raises(RuntimeError, match="checksum"):
        audit_source(source, root, parameters())
'''.lstrip()

print("Prepared focused source-audit tests.")



In [ ]:
for relative, source in new_sources.items():
    path = REPO_ROOT / relative
    if path.exists():
        existing = path.read_text(encoding="utf-8").replace("\r\n", "\n")
        if existing != source:
            raise RuntimeError(f"Existing source differs: {relative}. Stop.")

for relative, source in new_sources.items():
    path = REPO_ROOT / relative
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        with path.open("x", encoding="utf-8", newline="\n") as handle:
            handle.write(source)
        print("CREATED", relative)
    else:
        print("REUSED", relative)

importlib.invalidate_caches()
probe = subprocess.run(
    [
        sys.executable, "-c",
        "from pathlib import Path; from eurusd_research import coverage_audit; "
        "print(Path(coverage_audit.__file__).resolve())",
    ],
    cwd=REPO_ROOT / "notebooks",
    capture_output=True, text=True,
)
if probe.returncode:
    print(probe.stdout)
    print(probe.stderr)
    raise RuntimeError("Coverage-audit import failed.")
if Path(probe.stdout.strip()).resolve() != (
    REPO_ROOT / "src/eurusd_research/coverage_audit.py"
).resolve():
    raise RuntimeError("Imported a different coverage-audit module.")

test_run = subprocess.run(
    [sys.executable, "-m", "pytest", "tests", "-q", "--color=no"],
    cwd=REPO_ROOT, capture_output=True, text=True,
)
print(test_run.stdout)
if test_run.stderr:
    print(test_run.stderr)
if test_run.returncode:
    raise RuntimeError("Tests failed. Stop and return Cell 12 output.")
test_summary = test_run.stdout.strip().splitlines()[-1]

for relative in (
    "data/interim/coverage_audit/example.parquet",
    "artifacts/manifests/coverage_audit/example.json",
    "reports/private/notebook07_execution.json",
    "reports/tables/coverage_audit/example.csv",
    "reports/figures/coverage_audit/example.png",
):
    checked = subprocess.run(
        ["git", "check-ignore", "--no-index", "-q", "--", relative],
        cwd=REPO_ROOT, capture_output=True, text=True,
    )
    if checked.returncode != 0:
        raise RuntimeError(f"Generated path is not ignored: {relative}")

from eurusd_research.coverage_audit import (
    COUNT_COLUMNS, GAP_LABELS, audit_source, source_overlap_ranges,
)

print("PASS: imports, tests, and output ignore rules.")

In [ ]:
synthetic_path = REPO_ROOT / "data/sample/synthetic_quotes.csv"
if not synthetic_path.is_file():
    raise RuntimeError("The existing synthetic sample is missing.")

synthetic_audit = audit_source(
    synthetic_path, REPO_ROOT, audit_parameters
)
assert synthetic_audit["rows_scanned"] == (
    synthetic_audit["valid_timestamp_rows"]
    + synthetic_audit["unassigned_timestamp_rows"]
)

print("SYNTHETIC_COVERAGE_AUDIT")
print({
    "rows_scanned": synthetic_audit["rows_scanned"],
    "valid_timestamp_rows": synthetic_audit["valid_timestamp_rows"],
    "unassigned_timestamp_rows": synthetic_audit["unassigned_timestamp_rows"],
    "adjacent_exact_export_duplicate_rows":
        synthetic_audit["adjacent_exact_export_duplicate_rows"],
    "cache_hit": synthetic_audit["cache_hit"],
})
print("PASS: synthetic row conservation and saved-output integrity.")

In [ ]:
RUN_FULL_SOURCE_AUDIT = True

if not RUN_FULL_SOURCE_AUDIT:
    raise RuntimeError("Full-source audit disabled; later cells require its results.")
if shutil.disk_usage(REPO_ROOT).free < 1024**3:
    raise RuntimeError("Free at least 1 GiB before saving audit summaries.")

audit_started = time.perf_counter()
source_audits = []

for number, path in enumerate(source_files, start=1):
    if _stamp(path) != initial_stamps[path.name]:
        raise RuntimeError(f"Source changed since inventory: {path.name}")

    print(f"\nSOURCE {number}/{len(source_files)}: {path.name}")
    state = audit_source(path, REPO_ROOT, audit_parameters)
    source_audits.append(state)

    print({
        "rows_scanned": state["rows_scanned"],
        "valid_timestamp_rows": state["valid_timestamp_rows"],
        "unassigned_timestamp_rows": state["unassigned_timestamp_rows"],
        "utc_min": state["utc_min"],
        "utc_max": state["utc_max"],
        "cache_hit": state["cache_hit"],
    })

current_sources = matching_sources(RAW_ROOT)
if [path.name for path in current_sources] != [path.name for path in source_files]:
    raise RuntimeError(
        "The source inventory changed. Rerun from Cell 2; completed audits can be reused."
    )
for path in source_files:
    if _stamp(path) != initial_stamps[path.name]:
        raise RuntimeError(f"Source changed during the collection audit: {path.name}")

audit_runtime = time.perf_counter() - audit_started

print("\nPASS: every selected source audit completed.")
print("Audit runtime this execution:", round(audit_runtime, 3), "seconds")

In [ ]:
source_records = []
daily_pieces = []
global_flags = {}
largest_gap_records = []

for state in source_audits:
    source_records.append({
        "source_file": state["source_name"],
        "source_sha256": state["source_sha256"],
        "source_bytes": state["identity"]["source_bytes"],
        "rows_scanned": state["rows_scanned"],
        "valid_timestamp_rows": state["valid_timestamp_rows"],
        "unassigned_timestamp_rows": state["unassigned_timestamp_rows"],
        "price_eligible_quote_rows": state["price_eligible_quote_rows"],
        "size_eligible_quote_rows": state["size_eligible_quote_rows"],
        "utc_min": state["utc_min"],
        "utc_max": state["utc_max"],
        "adjacent_exact_export_duplicate_rows":
            state["adjacent_exact_export_duplicate_rows"],
        "source_order_backsteps": state["source_order_backsteps"],
        "repeated_valid_timestamp_steps": state["repeated_valid_timestamp_steps"],
        "gaps_above_threshold": state["gaps_above_threshold"],
        "maximum_source_sequence_gap_seconds":
            state["maximum_source_sequence_gap_seconds"],
        "recorded_source_content_versions":
            state["recorded_source_content_versions"],
    })
    daily_pieces.append(pd.read_parquet(
        REPO_ROOT / state["daily_output"]["path"]
    ))
    for name, count in state["flags"].items():
        global_flags[name] = global_flags.get(name, 0) + int(count)
    for gap in state["largest_gaps"]:
        largest_gap_records.append({
            "source_file": state["source_name"], **gap,
        })

source_table = pd.DataFrame(source_records)
per_source_daily = pd.concat(daily_pieces, ignore_index=True)
if per_source_daily.empty:
    raise RuntimeError("No assignable timestamps exist in the selected sources.")

aggregation = {name: "sum" for name in COUNT_COLUMNS}
aggregation.update({
    "first_quote_utc": "min",
    "last_quote_utc": "max",
    "min_spread_pips": "min",
    "max_spread_pips": "max",
    "max_gap_seconds": "max",
    "source_file": "nunique",
})
daily_table = per_source_daily.groupby("date_utc").agg(aggregation).rename(
    columns={"source_file": "sources_contributing"}
)

global_first = pd.to_datetime(source_table["utc_min"], utc=True).min()
global_last = pd.to_datetime(source_table["utc_max"], utc=True).max()

calendar = pd.date_range(
    global_first.normalize(), global_last.normalize(), freq="D"
)
daily_table = daily_table.reindex(calendar)
daily_table.index.name = "date_utc"

for name in COUNT_COLUMNS + ["sources_contributing"]:
    daily_table[name] = daily_table[name].fillna(0).astype("int64")

daily_table["zero_quote_day"] = daily_table["quote_rows"].eq(0)
daily_table["utc_weekday"] = daily_table.index.dayofweek
daily_table["zero_quote_weekday"] = (
    daily_table["zero_quote_day"] & daily_table["utc_weekday"].lt(5)
)
daily_table = daily_table.reset_index()

monthly_table = daily_table.assign(
    month_utc=daily_table["date_utc"].dt.strftime("%Y-%m")
).groupby("month_utc", as_index=False).agg(
    quote_rows=("quote_rows", "sum"),
    price_eligible_quote_rows=("price_eligible_quote_rows", "sum"),
    size_eligible_quote_rows=("size_eligible_quote_rows", "sum"),
    days_with_quotes=("quote_rows", lambda values: int(values.gt(0).sum())),
    zero_quote_weekdays=("zero_quote_weekday", "sum"),
    wide_spread_rows=("wide_spread_rows", "sum"),
    crossed_quote_rows=("crossed_quote_rows", "sum"),
    maximum_source_sequence_gap_seconds=("max_gap_seconds", "max"),
)

year_table = daily_table.assign(
    year_utc=daily_table["date_utc"].dt.year
).groupby("year_utc", as_index=False).agg(
    quote_rows=("quote_rows", "sum"),
    price_eligible_quote_rows=("price_eligible_quote_rows", "sum"),
    size_eligible_quote_rows=("size_eligible_quote_rows", "sum"),
    days_with_quotes=("quote_rows", lambda values: int(values.gt(0).sum())),
    zero_quote_weekdays=("zero_quote_weekday", "sum"),
    wide_spread_rows=("wide_spread_rows", "sum"),
    crossed_quote_rows=("crossed_quote_rows", "sum"),
)

overlap_records = source_overlap_ranges(source_audits)
overlap_table = pd.DataFrame(overlap_records, columns=[
    "left_source", "right_source",
    "overlap_start_utc", "overlap_end_utc",
    "overlap_seconds", "shared_boundary_timestamp_only",
    "identical_records_proven",
])

ordered_sources = sorted(
    [state for state in source_audits if state["utc_min"] is not None],
    key=lambda state: pd.Timestamp(state["utc_min"]),
)
boundary_records = []
for left, right in zip(ordered_sources, ordered_sources[1:]):
    boundary_records.append({
        "left_source": left["source_name"],
        "right_source": right["source_name"],
        "left_max_utc": left["utc_max"],
        "right_min_utc": right["utc_min"],
        "range_boundary_difference_seconds": float(
            (pd.Timestamp(right["utc_min"]) - pd.Timestamp(left["utc_max"]))
            .total_seconds()
        ),
        "last_first_export_records_identical": (
            left["last_record"]["raw_fields"]
            == right["first_record"]["raw_fields"]
        ),
    })
boundary_table = pd.DataFrame(boundary_records)

duplicate_content_table = source_table.loc[
    source_table["source_sha256"].duplicated(keep=False),
    ["source_file", "source_sha256"],
]
zero_day_table = daily_table.loc[daily_table["zero_quote_day"]].copy()
largest_gap_table = pd.DataFrame(largest_gap_records).sort_values(
    ["seconds", "source_file"], ascending=[False, True]
).head(50).reset_index(drop=True) if largest_gap_records else pd.DataFrame()

assert int(source_table["rows_scanned"].sum()) == (
    int(source_table["valid_timestamp_rows"].sum())
    + int(source_table["unassigned_timestamp_rows"].sum())
)
assert int(daily_table["quote_rows"].sum()) == int(
    source_table["valid_timestamp_rows"].sum()
)

print("ACTUAL_SOURCE_COVERAGE")
print(source_table[[
    "source_file", "rows_scanned", "unassigned_timestamp_rows",
    "utc_min", "utc_max", "source_order_backsteps",
]].to_string(index=False))

print("\nYEARLY_QUOTE_COVERAGE")
print(year_table.to_string(index=False))
print("\nPOSSIBLE_SOURCE_OVERLAPS")
print(overlap_table.to_string(index=False) if len(overlap_table) else "(none)")
print("\nSOURCE_BOUNDARIES")
print(boundary_table.to_string(index=False))
print("\nPASS: source and daily row counts reconcile.")

In [ ]:
feature_calibration = json.loads(
    (REPO_ROOT / config["active_features_manifest"]).read_text(encoding="utf-8")
)
label_calibration = json.loads(
    (REPO_ROOT / config["active_labels_manifest"]).read_text(encoding="utf-8")
)
ingestion_calibration = json.loads(
    (REPO_ROOT / config["active_ingestion_manifest"]).read_text(encoding="utf-8")
)

feature_rows = sum(part["rows"] for part in feature_calibration["parts"])
label_rows = sum(part["rows"] for part in label_calibration["parts"])
quote_rows = ingestion_calibration["rows_in_scope"]

if min(feature_rows, label_rows, quote_rows) <= 0:
    raise RuntimeError("The existing storage calibration has no usable rows.")

feature_bytes_per_row = (
    sum(part["bytes"] for part in feature_calibration["parts"]) / feature_rows
)
label_bytes_per_row = (
    sum(part["bytes"] for part in label_calibration["parts"]) / label_rows
)
quote_bytes_per_row = (
    sum(
        part["bytes"] for part in ingestion_calibration["parts"]
        if part["kind"] == "quotes"
    ) / quote_rows
)

projected_clock_rows = int((
    global_last.normalize() + pd.Timedelta(days=1)
    - global_first.normalize()
).total_seconds())
assignable_source_rows = int(source_table["valid_timestamp_rows"].sum())

storage_table = pd.DataFrame([
    {
        "output": "canonical_quotes",
        "projected_rows": assignable_source_rows,
        "january_bytes_per_row": quote_bytes_per_row,
        "projected_bytes": assignable_source_rows * quote_bytes_per_row,
    },
    {
        "output": "one_second_features",
        "projected_rows": projected_clock_rows,
        "january_bytes_per_row": feature_bytes_per_row,
        "projected_bytes": projected_clock_rows * feature_bytes_per_row,
    },
    {
        "output": "forward_labels",
        "projected_rows": projected_clock_rows,
        "january_bytes_per_row": label_bytes_per_row,
        "projected_bytes": projected_clock_rows * label_bytes_per_row,
    },
])
storage_table["projected_gib"] = storage_table["projected_bytes"] / 1024**3

free_bytes = shutil.disk_usage(REPO_ROOT).free
planning_storage_bytes = (
    float(storage_table["projected_bytes"].sum())
    * audit_settings["output_estimate_multiplier"]
    + audit_settings["scratch_reserve_gib"] * 1024**3
)

predictor_count = (
    len(config["baseline_models"]["price_features"])
    + len(config["baseline_models"]["size_features"])
)
training_memory_table = pd.DataFrame([
    {
        "training_window_days": days,
        "calendar_rows_upper_bound": days * 86400,
        "predictor_count": predictor_count,
        "single_predictor_matrix_gib": days * 86400 * predictor_count * 8 / 1024**3,
        "matrix_with_indicator_for_every_predictor_gib":
            days * 86400 * predictor_count * 2 * 8 / 1024**3,
    }
    for days in audit_settings["planning_training_windows_days"]
])

scaling_plan = {
    "status": "coverage audit complete; combination policy and dates require review",
    "observed_start_utc": global_first.isoformat(),
    "observed_end_utc": global_last.isoformat(),
    "full_calendar_grid_rows_for_observed_date_range": projected_clock_rows,
    "planning_training_windows_days":
        audit_settings["planning_training_windows_days"],
    "full_history_final_test_dates": None,
    "next_processing_stage": [
        "Review timestamp and ordering issues.",
        "Resolve source overlap and duplicate-handling policy.",
        "Ingest canonical multi-source quotes with provenance.",
        "Generate boundary-consistent features and labels.",
        "Benchmark a longer training sample before increasing window size.",
        "Select chronological development and untouched final-test periods.",
    ],
    "h001_status": "pending; no dedicated test executed",
    "storage_planning_bytes_including_multiplier_and_scratch":
        int(planning_storage_bytes),
    "available_disk_bytes": int(free_bytes),
    "disk_estimate_fits_current_free_space": planning_storage_bytes <= free_bytes,
}

print("PROJECTED_GENERATED_STORAGE")
print(storage_table.to_string(index=False))
print("Planning storage GiB including reserve:",
      round(planning_storage_bytes / 1024**3, 2))
print("Current free disk GiB:", round(free_bytes / 1024**3, 2))
print("\nDENSE_MATRIX_PLANNING — not peak fitting memory")
print(training_memory_table.to_string(index=False))
print("No longer-window model has been fitted.")

In [ ]:
import matplotlib.pyplot as plt

figure, axes = plt.subplots(2, 1, figsize=(11, 7), constrained_layout=True)

axes[0].bar(
    year_table["year_utc"].astype(str),
    year_table["quote_rows"] / 1_000_000,
    color="#3B6EA8",
)
axes[0].set_ylabel("Quote records (millions)")
axes[0].set_title("Raw export coverage by UTC year — boundary years may be partial")
axes[0].grid(axis="y", alpha=0.25)

axes[1].bar(
    year_table["year_utc"].astype(str),
    year_table["zero_quote_weekdays"],
    color="#D68132",
)
axes[1].set_ylabel("Zero-quote UTC weekdays")
axes[1].set_xlabel("UTC year")
axes[1].set_title("Review candidates; holidays and outages are not distinguished")
axes[1].grid(axis="y", alpha=0.25)

collection_identity = {
    "source_audits": [
        {
            "source_file": state["source_name"],
            "source_sha256": state["source_sha256"],
            "audit_job_id": state["job_id"],
            "audit_manifest_sha256": _sha256(REPO_ROOT / state["manifest_path"]),
        }
        for state in source_audits
    ],
    "audit_parameters": audit_parameters,
    "planning_settings": audit_settings,
    "versions": versions,
}
collection_id = hashlib.sha256(
    json.dumps(collection_identity, sort_keys=True).encode("utf-8")
).hexdigest()[:24]

figure_path = (
    REPO_ROOT / "reports/figures" / f"history-coverage-{collection_id}.png"
)
figure_path.parent.mkdir(parents=True, exist_ok=True)
figure.savefig(figure_path, dpi=160)
plt.show()
plt.close(figure)

print("Saved:", figure_path.relative_to(REPO_ROOT).as_posix())

In [ ]:
def relative_path(path):
    return Path(path).resolve().relative_to(REPO_ROOT).as_posix()


def update_section(path, name, content):
    begin = f"<!-- {name}:BEGIN -->"
    finish = f"<!-- {name}:END -->"
    existing = path.read_text(encoding="utf-8") if path.exists() else ""
    if existing.count(begin) != existing.count(finish) or existing.count(begin) > 1:
        raise RuntimeError(f"Malformed generated section: {path.name}")
    block = f"{begin}\n{content.strip()}\n{finish}"
    if begin in existing:
        left, remainder = existing.split(begin, 1)
        _, right = remainder.split(finish, 1)
        text = left + block + right
    else:
        text = existing.rstrip() + "\n\n" + block + "\n"
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="\n") as handle:
        handle.write(text)


table_dir = REPO_ROOT / "reports/tables" / collection_id
table_dir.mkdir(parents=True, exist_ok=True)

tables = {
    "source_inventory": inventory_table,
    "source_audit_summary": source_table,
    "daily_by_source": per_source_daily,
    "daily_combined_export_counts": daily_table,
    "monthly_coverage": monthly_table,
    "yearly_coverage": year_table,
    "possible_source_overlaps": overlap_table,
    "source_boundaries": boundary_table,
    "identical_source_content": duplicate_content_table,
    "zero_quote_days": zero_day_table,
    "largest_source_sequence_gaps": largest_gap_table,
    "storage_projection": storage_table,
    "training_matrix_projection": training_memory_table,
}
table_paths = {}
output_records = []

for name, table in tables.items():
    path = table_dir / f"{name}.csv"
    table.to_csv(path, index=False)
    table_paths[name] = relative_path(path)
    output_records.append({
        "path": relative_path(path),
        "bytes": path.stat().st_size,
        "sha256": _sha256(path),
    })

scaling_plan_path = (
    REPO_ROOT / "artifacts/manifests/coverage_audit"
    / f"scale-plan-{collection_id}.json"
)
feature_io.write_json(scaling_plan_path, scaling_plan)

collection_manifest_path = (
    REPO_ROOT / "artifacts/manifests/coverage_audit"
    / f"collection-{collection_id}.json"
)
collection_state = {
    "collection_id": collection_id,
    "status": "complete",
    "identity": collection_identity,
    "source_manifests": [state["manifest_path"] for state in source_audits],
    "outputs": output_records,
    "scaling_plan": relative_path(scaling_plan_path),
    "validation": {
        "pytest": test_summary,
        "source_integrity": "verified before and after each source scan",
        "source_and_daily_row_conservation": "passed",
        "overlap_resolution": "pending review",
        "owner_review": "pending",
        "predictive_holdout_evaluated": False,
    },
}
feature_io.write_json(collection_manifest_path, collection_state)

example_path = REPO_ROOT / "configs/research_config.example.json"
local_path = REPO_ROOT / "configs/research_config.local.json"
example = json.loads(example_path.read_text(encoding="utf-8-sig"))
local = json.loads(local_path.read_text(encoding="utf-8-sig"))

example["full_history_audit"] = audit_settings
local["active_coverage_audit_manifest"] = relative_path(collection_manifest_path)
local["raw_data_dir"] = (
    "data" if RAW_ROOT == (REPO_ROOT / "data").resolve() else str(RAW_ROOT)
)
feature_io.write_json(example_path, example)
feature_io.write_json(local_path, local)

environment_path = REPO_ROOT / "reports/private/validated_coverage_audit_environment.json"
feature_io.write_json(environment_path, {
    "python": platform.python_version(),
    "packages": versions,
    "validation": test_summary,
})

method_note = f"""
### Notebook 07 — full-history coverage and scaling preparation

Completed bounded-memory source audits for {len(source_audits)} selected exports.
The audit measures actual UTC coverage, daily/yearly record counts,
quality flags, source-order gaps, adjacent exact-export duplicates,
and possible cross-file range overlaps.

Original CSVs are preserved. Changed source content has a distinct audit
identity. Completed audit outputs are verified before reuse.

Wide spreads and market-closure gaps are reported without automatic exclusion.
Source-range overlaps do not prove identical records.

Executed correctness checks: {test_summary}.
Source integrity and daily/source row conservation passed.

The research sequence now includes full-history coverage and scale planning
before final interpretation. Multi-source combination, longer-window
training, multi-year evaluation, and untouched final testing remain pending.

Resource projections are planning estimates, not measured full-history
output sizes or peak-memory benchmarks.

H001 remains untested. Notebook 07 owner review remains pending.
"""

for relative in (
    "README.md", "reports/data_quality.md", "reports/research_results.md",
):
    update_section(REPO_ROOT / relative, "NOTEBOOK_07", method_note)

update_section(
    REPO_ROOT / "CONTRIBUTIONS.md",
    "NOTEBOOK_07",
    f"""
### Notebook 07

- Raymond Fendi questioned the adequacy of the short pilot training windows
  and requested the next notebook.
- OpenAI Codex supplied the full-source coverage audit, focused tests,
  scaling estimates, and documentation.
- Executed correctness checks: {test_summary}.
- Source/daily conservation and source-integrity checks passed.
- Source issue resolution and owner review: pending.
- No full-history model or dedicated H001 test was executed.
""",
)

private_report_path = (
    REPO_ROOT / "reports/private" / f"coverage-audit-{collection_id}.md"
)
private_report_path.parent.mkdir(parents=True, exist_ok=True)
private_report_path.write_text(
    "# Full-source coverage audit\n\n"
    "Source audit completed; combination policy and final-test dates remain pending.\n\n"
    "## Source coverage\n\n```\n"
    + source_table.to_string(index=False)
    + "\n```\n\n## Possible overlaps\n\n```\n"
    + overlap_table.to_string(index=False)
    + "\n```\n\n## Resource planning\n\n```\n"
    + storage_table.to_string(index=False)
    + "\n```\n\nNo full-history training or predictive evaluation occurred.\n",
    encoding="utf-8",
)

print("Saved full-source coverage audit and scaling plan.")
print("Raw-data configuration now points to:", RAW_ROOT)
print("Source issue review and final-test date selection remain pending.")

In [ ]:
def json_records(frame):
    return json.loads(frame.to_json(orient="records", date_format="iso"))


source_summary_columns = [
    "source_file", "source_bytes", "rows_scanned",
    "valid_timestamp_rows", "unassigned_timestamp_rows",
    "price_eligible_quote_rows", "size_eligible_quote_rows",
    "utc_min", "utc_max",
    "adjacent_exact_export_duplicate_rows",
    "source_order_backsteps", "repeated_valid_timestamp_steps",
    "gaps_above_threshold",
    "maximum_source_sequence_gap_seconds",
    "recorded_source_content_versions",
]

gap_histogram = {
    label: sum(state["gap_histogram"][index] for state in source_audits)
    for index, label in enumerate(GAP_LABELS)
}
weekday_zero_dates = daily_table.loc[
    daily_table["zero_quote_weekday"], "date_utc"
].dt.strftime("%Y-%m-%d").tolist()

summary = {
    "notebook": NOTEBOOK,
    "collection_id": collection_id,
    "scope": "full selected raw-source coverage and quality audit",
    "python_version": platform.python_version(),
    "package_versions": versions,
    "configuration": {
        "source_files": len(source_files),
        "source_bytes": total_source_bytes,
        "audit_parameters": audit_parameters,
        "planning_settings": audit_settings,
    },
    "actual_coverage": {
        "first_mappable_timestamp_utc": global_first.isoformat(),
        "last_mappable_timestamp_utc": global_last.isoformat(),
        "calendar_dates_in_observed_range": len(daily_table),
        "dates_with_quotes": int(daily_table["quote_rows"].gt(0).sum()),
        "zero_quote_dates": int(daily_table["zero_quote_day"].sum()),
        "zero_quote_utc_weekdays": len(weekday_zero_dates),
        "first_zero_quote_weekday_examples": weekday_zero_dates[:15],
        "rows_scanned": int(source_table["rows_scanned"].sum()),
        "valid_timestamp_rows": int(source_table["valid_timestamp_rows"].sum()),
        "unassigned_timestamp_rows": int(source_table["unassigned_timestamp_rows"].sum()),
    },
    "sources": json_records(source_table[source_summary_columns]),
    "yearly_coverage": json_records(year_table),
    "global_source_quality_flags": global_flags,
    "source_sequence_gap_histogram_seconds": gap_histogram,
    "possible_source_overlaps": overlap_records,
    "source_boundaries": boundary_records,
    "identical_complete_source_content": json_records(duplicate_content_table),
    "largest_source_sequence_gap_examples": (
        json_records(largest_gap_table.head(10)) if len(largest_gap_table) else []
    ),
    "resource_planning": {
        "storage_projection": json_records(storage_table),
        "training_matrix_projection": json_records(training_memory_table),
        "planning_storage_bytes_including_reserve": int(planning_storage_bytes),
        "free_disk_bytes": int(free_bytes),
        "disk_estimate_fits": planning_storage_bytes <= free_bytes,
    },
    "validation": {
        "imports_from_notebooks_directory": "passed",
        "pytest": test_summary,
        "synthetic_pipeline": "passed",
        "source_hash_and_stamp_integrity": "passed",
        "source_and_daily_row_conservation": "passed",
        "adjacent_duplicate_and_gap_batch_consistency": "covered by tests",
        "global_exact_duplicate_detection": "not performed",
        "cross_file_overlap_resolution": "pending review",
        "predictive_holdout_evaluated": False,
        "full_history_models_fitted": False,
        "full_history_final_test_dates_selected": False,
        "owner_review": "pending",
    },
    "measurements": {
        "audit_seconds_this_execution": round(audit_runtime, 3),
        "sources_scanned_this_execution": sum(
            not state["cache_hit"] for state in source_audits
        ),
        "sources_reused_after_verification": sum(
            state["cache_hit"] for state in source_audits
        ),
        "maximum_observed_raw_plus_normalized_frame_bytes": max(
            state["maximum_observed_dataframe_bytes"] for state in source_audits
        ),
        "daily_audit_parquet_bytes": sum(
            state["daily_output"]["bytes"] for state in source_audits
        ),
        "notebook_elapsed_seconds": round(time.perf_counter() - notebook_started, 3),
    },
    "warnings": [
        "Export names are not verified coverage boundaries.",
        "File timestamp spans do not establish uninterrupted coverage.",
        "Combined counts include all exported records before overlap resolution.",
        "Repeated timestamps are not automatically duplicate observations.",
        "Only adjacent exact-export duplicates were counted within sources.",
        "Source-sequence gaps can include closures, outages, ordering issues, or unassigned rows.",
        "Zero-quote UTC weekdays require holiday/source review.",
        "Wide spreads are flags without automatic exclusion.",
        "Sustained-wide-spread episode validation remains pending.",
        "Timezone interpretation still has the previously documented verification limits.",
        "Unassigned timestamps cannot be allocated to a reliable UTC date.",
        "Storage extrapolations and matrix arithmetic are not full-history benchmarks.",
        "Observed dataframe sizes are not total process peak memory.",
        "H001 remains untested.",
        "No full-history predictive or profitability finding has been established.",
    ],
    "saved_outputs": {
        "active_coverage_audit_manifest": relative_path(collection_manifest_path),
        "scaling_plan": relative_path(scaling_plan_path),
        "tables": table_paths,
        "coverage_figure": relative_path(figure_path),
        "private_report": relative_path(private_report_path),
        "validated_environment": relative_path(environment_path),
        "execution_summary": "reports/private/notebook07_execution.json",
    },
}

feature_io.write_json(
    REPO_ROOT / "reports/private/notebook07_execution.json", summary
)
print("NOTEBOOK_07_SUMMARY")
print(json.dumps(summary, indent=2, allow_nan=False))